# Markdown management under demand uncertainty

This is the stochastic counterpart to the deterministic markdown notebook. The setup is the same — $x = 160$ units of seasonal inventory, salvage price $r = 5$, declining monthly demand curves — but now each month's demand carries an additive shock $\varepsilon \sim \mathcal{N}(0, 10^2)$. Because demand is random, we evaluate any pricing rule by its **expected** revenue, estimated by Monte Carlo over $N = 10{,}000$ draws.

We build up the same way as before:

1. A **single-period** benchmark: one price against the full inventory.
2. A **probabilistic markdown heuristic**: a single price applied across all four months' (random) demand.
3. **Adaptive re-optimization** after period 1's realized sales, re-solving the remaining periods with updated inventory.

Carrying the uncertainty explicitly is the point of Phillips' probabilistic algorithm (12.2.2): the deterministic version reacts to surprises only after they happen, whereas accounting for the noise up front leads to more robust prices.

## Single-period benchmark

A warm-up: one price against the entire inventory, with a single random demand curve $d(p, \varepsilon) = \max(120 - 1.5p + \varepsilon,\, 0)$. We maximize expected revenue (sales at $p$ plus salvage of leftovers at $r$) over the Monte Carlo draws.

In [1]:
import numpy as np
from scipy.optimize import minimize
np.random.seed(666)

In [2]:
x = 160   # initial inventory
r = 5     # salvage price after the fourth month
N = 10000  # Monte Carlo samples

epsilons = np.random.normal(0, 10, N)

def d(p, epsilon):
    return np.maximum(120 - 1.5 * p + epsilon, 0)

def total_revenue(p):
    demands = d(p, epsilons)
    q = np.minimum(demands, x)
    unsold = x - q
    revenues = p*q + r*unsold
    return -np.mean(revenues)   # negate expected revenue: maximize via a minimizer

res = minimize(total_revenue, 30, bounds=[(0, 120/1.5)])

print('The optimal price is:', round(res.x[0], 2))
print('The maximum expected revenue is:', round(-res.fun, 2))

The optimal price is: 42.44
The maximum expected revenue is: 2902.92


## Probabilistic markdown heuristic

Now use all four months, each with its own declining demand curve and an independent shock. The heuristic holds a single price across the season and measures it against total demand summed over the four months. Each month draws its own $\varepsilon$, so the realized demand path varies from run to run.

In [3]:
x = 160
r = 5
N = 10000

# Independent demand shocks for each of the four months
epsilons_1 = np.random.normal(0, 10, N)
epsilons_2 = np.random.normal(0, 10, N)
epsilons_3 = np.random.normal(0, 10, N)
epsilons_4 = np.random.normal(0, 10, N)

# Declining monthly demand curves
def d1(p, epsilon):
    return np.maximum(120 - 1.5 * p + epsilon, 0)

def d2(p, epsilon):
    return np.maximum(90 - 1.5 * p + epsilon, 0)

def d3(p, epsilon):
    return np.maximum(80 - 1.5 * p + epsilon, 0)

def d4(p, epsilon):
    return np.maximum(50 - 2 * p + epsilon, 0)

def total_revenue(p):
    D = d1(p, epsilons_1) + d2(p, epsilons_2) + d3(p, epsilons_3) + d4(p, epsilons_4)
    q = np.minimum(D, x)
    unsold = x - q
    revenues = p*q + r*unsold
    return -np.mean(revenues)

res = minimize(total_revenue, 30, bounds=[(0, 120/1.5)])

print('The optimal price is:', round(res.x[0], 2))
print('The maximum expected revenue is:', round(-res.fun, 2))

The optimal price is: 35.32
The maximum expected revenue is: 4766.77


## Re-optimizing after period 1

As in the deterministic case, the heuristic price is a starting plan. Once period 1's sales are realized — say $70$ units — we update the inventory and re-solve for the remaining three months, again maximizing expected revenue over fresh demand draws.

In [4]:
p_optimal = res.x[0]
q1_realized = 70
print('Say you ended up selling ', q1_realized, 'units.')

Say you ended up selling  70 units.


In [5]:
p1 = res.x[0]
q1 = 70
x2 = x - q1            # inventory carried into period 2
revenue1 = p1 * q1     # realized period-1 revenue

# Total demand over the remaining three months
def D(p, epsilons_2, epsilons_3, epsilons_4):
    return d2(p, epsilons_2) + d3(p, epsilons_3) + d4(p, epsilons_4)

def total_revenue(p):
    D_total = D(p, epsilons_2, epsilons_3, epsilons_4)
    q = np.minimum(D_total, x2)
    unsold = x2 - q
    revenues = p*q + r*unsold
    return -np.mean(revenues)

res = minimize(total_revenue, 30, bounds=[(0, 90/1.5)])

print('The optimal price for the remaining periods is:', round(res.x[0], 2))
print('The maximum expected revenue for the remaining periods is:', round(-res.fun, 2))
print('The total expected revenue, including the first period, is:', round(revenue1 - res.fun, 2))

The optimal price for the remaining periods is: 32.04
The maximum expected revenue for the remaining periods is: 2436.57
The total expected revenue, including the first period, is: 4908.83
